# Build 3 — Strong Model Benchmarks

Kaggle Playground Series S6E8 — Predicting Smartphone Addiction

**Purpose:** answer one question cleanly — which strong gradient-boosting
model family (CatBoost, LightGBM, XGBoost) performs best on the current
raw feature set, under the frozen Build 2 validation framework, and which
should become the primary control for Build 4 feature engineering.

**Out of scope for this build:** broad feature engineering, hyperparameter
search, ensembling/stacking, final submission strategy. E001 (Logistic
Regression) is treated as authoritative and is read from
`experiments/experiments.csv`, not rerun.

Each benchmark (E002 CatBoost, E003 LightGBM, E004 XGBoost) uses one
untuned, sensible configuration, evaluated with the same
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` harness used
for E001, on the same raw predictor set (`src.config.FEATURE_COLS`).

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.benchmarking import CVBenchmarkResult, run_cv_benchmark
from src.boosting_models import (
    EARLY_STOPPING_ROUNDS,
    LEARNING_RATE,
    MAX_ITERATIONS,
    catboost_fold,
    lightgbm_fold,
    xgboost_fold,
)
from src.config import (
    CATEGORICAL_COLS,
    DELIVERABLES_DIR,
    EXPERIMENTS_DIR,
    FEATURE_COLS,
    ID_COLUMN,
    NUMERIC_COLS,
    OUTPUTS_DIR,
    RANDOM_SEED,
    SAMPLE_SUBMISSION_PATH,
    TARGET_COLUMN,
    TEST_PATH,
    TRAIN_PATH,
)
from src.preprocessing import build_boosting_frame
from src.submission_validation import validate_submission
from src.validation import N_SPLITS

pd.set_option("display.max_columns", 50)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train shape:", train.shape)
print("test shape:", test.shape)
print(f"shared training budget: max_iterations={MAX_ITERATIONS}, "
      f"learning_rate={LEARNING_RATE}, early_stopping_rounds={EARLY_STOPPING_ROUNDS}")

train shape: (691369, 14)
test shape: (296302, 13)
shared training budget: max_iterations=800, learning_rate=0.1, early_stopping_rounds=50


## 2. Load the frozen raw feature set

`build_boosting_frame` (added to `src/preprocessing.py` in this build)
applies the same raw predictor set as E001 (`src.config.FEATURE_COLS`),
but leaves numeric missing values untouched — CatBoost, LightGBM, and
XGBoost all handle missing numeric values natively, so imputing here
would remove information these models can use directly. Categorical
missing values still get the explicit "Missing" category frozen for E001
(`docs/DECISIONS.md`), cast to a pandas `category` dtype so LightGBM and
XGBoost can split on it natively; CatBoost's `cat_features` accepts the
same columns directly.

This is the one place preprocessing legitimately differs from E001 — by
necessity of what these models can do, not as an engineered advantage for
any one model family (all three get the same treatment).

In [2]:
X = build_boosting_frame(train)
y = train[TARGET_COLUMN]
X_test = build_boosting_frame(test)

print("feature columns:", list(X.columns))
print("numeric missing counts (preserved):")
print(X[NUMERIC_COLS].isna().sum())
print("categorical dtypes (should be 'category'):")
print(X[CATEGORICAL_COLS].dtypes)

feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']
numeric missing counts (preserved):
age                         28929
daily_screen_time_hours     95854
social_media_hours         133995
gaming_hours               126821
work_study_hours            51518
sleep_hours                 44480
notifications_per_day       67584
app_opens_per_day           80710
weekend_screen_time        112063
dtype: int64
categorical dtypes (should be 'category'):
gender                  category
stress_level            category
academic_work_impact    category
dtype: object


## 3. E001 reference (read, not rerun)

Per Build 3 scope, E001 is authoritative and is read directly from
`experiments/experiments.csv` rather than rewritten or rerun here.

In [3]:
experiments_path = EXPERIMENTS_DIR / "experiments.csv"
experiments = pd.read_csv(experiments_path)

e001 = experiments.loc[experiments["experiment_id"] == "E001"].iloc[0]
E001_CV_MEAN = float(e001["cv_mean"])
E001_CV_STD = float(e001["cv_std"])
E001_PUBLIC_LB = float(e001["public_lb"]) if pd.notna(e001["public_lb"]) else None

print(f"E001 (LogisticRegression) CV mean: {E001_CV_MEAN:.5f}, CV std: {E001_CV_STD:.5f}")
print(f"E001 public LB: {E001_PUBLIC_LB}")

E001 (LogisticRegression) CV mean: 0.91149, CV std: 0.00081
E001 public LB: 0.91358


## 4. E002 — CatBoost raw benchmark

**Hypothesis:** CatBoost should substantially outperform the linear
baseline if meaningful nonlinear interactions and categorical
relationships exist in the dataset (Build 1 found a strongly nonlinear,
near-S-curve relationship between `daily_screen_time_hours` and the
target — Section 8 of `notebooks/01_eda.ipynb`). Its native categorical
and numerical missing-value handling may also suit this dataset well.

**Configuration:** `loss_function=Logloss`, `eval_metric=AUC`,
`iterations=800` (shared budget), `learning_rate=0.1`, `depth=6`,
`l2_leaf_reg=3.0`, `early_stopping_rounds=50`, native categorical handling
via `cat_features`, `allow_writing_files=False`. This is one untuned,
sensible configuration — not a search. The 800-iteration budget was
chosen for runtime practicality after a timing check found CatBoost still
improving at 2000 iterations; this is noted as a resource-discipline
tradeoff, not a claim that CatBoost is fully converged (see Section 10).

CPU only — no GPU is available in this environment (verified: no
`nvidia-smi`).

In [4]:
t0 = time.time()
e002_result = run_cv_benchmark(catboost_fold, X, y, X_test=X_test)
e002_elapsed = time.time() - t0

print()
print(f"CV mean ROC AUC: {e002_result.cv_mean:.5f}")
print(f"CV std:          {e002_result.cv_std:.5f}")
print(f"best_iterations: {e002_result.best_iterations}")
print(f"elapsed:         {e002_elapsed:.1f}s")

fold 1: ROC AUC = 0.95982


fold 2: ROC AUC = 0.96010


fold 3: ROC AUC = 0.96057


fold 4: ROC AUC = 0.96130


fold 5: ROC AUC = 0.96020

CV mean ROC AUC: 0.96040
CV std:          0.00051
best_iterations: [799, 799, 799, 799, 799]
elapsed:         2663.1s


## 5. E003 — LightGBM raw benchmark

**Hypothesis:** LightGBM should capture the same nonlinear structure
efficiently and may provide equal or stronger ranking performance than
CatBoost on this large tabular dataset.

**Configuration:** `objective=binary`, `metric=auc`,
`n_estimators=800` (shared budget), `learning_rate=0.1`, `num_leaves=31`,
`min_child_samples=20`, `feature_fraction=0.9`, `bagging_fraction=0.9`,
`bagging_freq=1`, `early_stopping_rounds=50`, native categorical handling
via pandas `category` dtype. No target encoding, no leakage-prone
encodings — same raw semantic information available to CatBoost.

In [5]:
t0 = time.time()
e003_result = run_cv_benchmark(lightgbm_fold, X, y, X_test=X_test)
e003_elapsed = time.time() - t0

print()
print(f"CV mean ROC AUC: {e003_result.cv_mean:.5f}")
print(f"CV std:          {e003_result.cv_std:.5f}")
print(f"best_iterations: {e003_result.best_iterations}")
print(f"elapsed:         {e003_elapsed:.1f}s")

fold 1: ROC AUC = 0.96036


fold 2: ROC AUC = 0.95927


fold 3: ROC AUC = 0.96133


fold 4: ROC AUC = 0.96235


fold 5: ROC AUC = 0.96201

CV mean ROC AUC: 0.96106
CV std:          0.00113
best_iterations: [388, 190, 322, 439, 634]
elapsed:         141.9s


## 6. E004 — XGBoost raw benchmark

**Hypothesis:** XGBoost provides another strong boosting implementation
whose regularization and tree-building behavior may produce different
ranking performance from CatBoost and LightGBM.

**Configuration:** `objective=binary:logistic`, `eval_metric=auc`,
`n_estimators=800` (shared budget), `learning_rate=0.1`, `max_depth=6`,
`min_child_weight=1`, `subsample=0.9`, `colsample_bytree=0.9`,
`reg_lambda=1.0`, `early_stopping_rounds=50`, `tree_method=hist`, native
categorical handling via `enable_categorical=True` + pandas `category`
dtype. No target encoding, no feature engineering.

In [6]:
t0 = time.time()
e004_result = run_cv_benchmark(xgboost_fold, X, y, X_test=X_test)
e004_elapsed = time.time() - t0

print()
print(f"CV mean ROC AUC: {e004_result.cv_mean:.5f}")
print(f"CV std:          {e004_result.cv_std:.5f}")
print(f"best_iterations: {e004_result.best_iterations}")
print(f"elapsed:         {e004_elapsed:.1f}s")

fold 1: ROC AUC = 0.96308


fold 2: ROC AUC = 0.96383


fold 3: ROC AUC = 0.96403


fold 4: ROC AUC = 0.96474


fold 5: ROC AUC = 0.96344

CV mean ROC AUC: 0.96382
CV std:          0.00056
best_iterations: [799, 794, 799, 794, 794]
elapsed:         571.6s


## 7. Recording E002-E004

Experiment IDs are assigned only when an experiment actually runs (per
`docs/DECISIONS.md`). All three have now run, so their results are
appended to `experiments/experiments.csv` as real rows.

In [7]:
import csv
from datetime import date

def append_experiment_row(row: dict) -> None:
    existing = pd.read_csv(experiments_path)
    if row["experiment_id"] in existing["experiment_id"].astype(str).values:
        print(f"{row['experiment_id']} already recorded — skipping append (idempotent re-run).")
        return
    with open(experiments_path, "r", newline="", encoding="utf-8") as f:
        fieldnames = next(csv.reader(f))
    with open(experiments_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
    print(f"Row appended for {row['experiment_id']}")


CV_METHOD = f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_SEED})"
SHARED_PREPROCESSING = (
    "raw numeric predictors (missing values preserved, native handling); "
    "categorical predictors: explicit Missing category, category dtype "
    "(native handling); id excluded"
)

append_experiment_row({
    "experiment_id": "E002",
    "date": date.today().isoformat(),
    "model": "CatBoostClassifier",
    "feature_set": "raw_predictors",
    "hypothesis": (
        "CatBoost should substantially outperform the linear baseline given "
        "Build 1's nonlinear screen-time/target relationship, and its native "
        "categorical/missing handling may suit this dataset well."
    ),
    "preprocessing": SHARED_PREPROCESSING + "; CatBoost cat_features=native",
    "cv_method": CV_METHOD,
    "seed": RANDOM_SEED,
    "parameters": (
        f"CatBoostClassifier(loss_function=Logloss, eval_metric=AUC, "
        f"iterations={MAX_ITERATIONS}, learning_rate={LEARNING_RATE}, depth=6, "
        f"l2_leaf_reg=3.0, early_stopping_rounds={EARLY_STOPPING_ROUNDS}, "
        f"random_seed={RANDOM_SEED})"
    ),
    "fold_1_auc": round(e002_result.fold_scores[0], 5),
    "fold_2_auc": round(e002_result.fold_scores[1], 5),
    "fold_3_auc": round(e002_result.fold_scores[2], 5),
    "fold_4_auc": round(e002_result.fold_scores[3], 5),
    "fold_5_auc": round(e002_result.fold_scores[4], 5),
    "cv_mean": round(e002_result.cv_mean, 5),
    "cv_std": round(e002_result.cv_std, 5),
    "public_lb": "",
    "submission_file": "",
    "conclusion": (
        f"CV mean {e002_result.cv_mean:.5f} vs E001 {E001_CV_MEAN:.5f} "
        f"(delta {e002_result.cv_mean - E001_CV_MEAN:+.5f}). "
        f"best_iterations={e002_result.best_iterations}, elapsed={e002_elapsed:.0f}s."
    ),
    "next_action": "Compare against E003/E004; see Section 9-10 for model selection.",
})

append_experiment_row({
    "experiment_id": "E003",
    "date": date.today().isoformat(),
    "model": "LGBMClassifier",
    "feature_set": "raw_predictors",
    "hypothesis": (
        "LightGBM should capture the same nonlinear structure efficiently and "
        "may match or exceed CatBoost's ranking performance on this dataset."
    ),
    "preprocessing": SHARED_PREPROCESSING + "; LightGBM categorical_feature=native",
    "cv_method": CV_METHOD,
    "seed": RANDOM_SEED,
    "parameters": (
        f"LGBMClassifier(objective=binary, metric=auc, "
        f"n_estimators={MAX_ITERATIONS}, learning_rate={LEARNING_RATE}, "
        f"num_leaves=31, min_child_samples=20, feature_fraction=0.9, "
        f"bagging_fraction=0.9, bagging_freq=1, "
        f"early_stopping_rounds={EARLY_STOPPING_ROUNDS}, random_state={RANDOM_SEED})"
    ),
    "fold_1_auc": round(e003_result.fold_scores[0], 5),
    "fold_2_auc": round(e003_result.fold_scores[1], 5),
    "fold_3_auc": round(e003_result.fold_scores[2], 5),
    "fold_4_auc": round(e003_result.fold_scores[3], 5),
    "fold_5_auc": round(e003_result.fold_scores[4], 5),
    "cv_mean": round(e003_result.cv_mean, 5),
    "cv_std": round(e003_result.cv_std, 5),
    "public_lb": "",
    "submission_file": "",
    "conclusion": (
        f"CV mean {e003_result.cv_mean:.5f} vs E001 {E001_CV_MEAN:.5f} "
        f"(delta {e003_result.cv_mean - E001_CV_MEAN:+.5f}). "
        f"best_iterations={e003_result.best_iterations}, elapsed={e003_elapsed:.0f}s."
    ),
    "next_action": "Compare against E002/E004; see Section 9-10 for model selection.",
})

append_experiment_row({
    "experiment_id": "E004",
    "date": date.today().isoformat(),
    "model": "XGBClassifier",
    "feature_set": "raw_predictors",
    "hypothesis": (
        "XGBoost's regularization and tree-building behavior may produce "
        "different ranking performance from CatBoost and LightGBM."
    ),
    "preprocessing": SHARED_PREPROCESSING + "; XGBoost enable_categorical=True",
    "cv_method": CV_METHOD,
    "seed": RANDOM_SEED,
    "parameters": (
        f"XGBClassifier(objective=binary:logistic, eval_metric=auc, "
        f"n_estimators={MAX_ITERATIONS}, learning_rate={LEARNING_RATE}, "
        f"max_depth=6, min_child_weight=1, subsample=0.9, colsample_bytree=0.9, "
        f"reg_lambda=1.0, early_stopping_rounds={EARLY_STOPPING_ROUNDS}, "
        f"tree_method=hist, random_state={RANDOM_SEED})"
    ),
    "fold_1_auc": round(e004_result.fold_scores[0], 5),
    "fold_2_auc": round(e004_result.fold_scores[1], 5),
    "fold_3_auc": round(e004_result.fold_scores[2], 5),
    "fold_4_auc": round(e004_result.fold_scores[3], 5),
    "fold_5_auc": round(e004_result.fold_scores[4], 5),
    "cv_mean": round(e004_result.cv_mean, 5),
    "cv_std": round(e004_result.cv_std, 5),
    "public_lb": "",
    "submission_file": "",
    "conclusion": (
        f"CV mean {e004_result.cv_mean:.5f} vs E001 {E001_CV_MEAN:.5f} "
        f"(delta {e004_result.cv_mean - E001_CV_MEAN:+.5f}). "
        f"best_iterations={e004_result.best_iterations}, elapsed={e004_elapsed:.0f}s."
    ),
    "next_action": "Compare against E002/E003; see Section 9-10 for model selection.",
})

pd.read_csv(experiments_path)

Row appended for E002
Row appended for E003
Row appended for E004


,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
0,E001,2026-08-17,LogisticRegression,raw_predictors,A linear baseline on imputed/standardized/one-...,median imputation + StandardScaler (numeric); ...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"LogisticRegression(max_iter=1000, random_state...",0.91040,0.91082,0.91193,0.91267,0.91161,0.91149,0.00081,0.91358,deliverables/E001_submission.csv,"Stable, credible linear baseline (mean 0.9115,...","Run E002 (missing indicators), E003 (screen-ti..."
1,E002,2026-08-17,CatBoostClassifier,raw_predictors,CatBoost should substantially outperform the l...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"CatBoostClassifier(loss_function=Logloss, eval...",0.95982,0.96010,0.96057,0.96130,0.96020,0.96040,0.00051,NaN,NaN,CV mean 0.96040 vs E001 0.91149 (delta +0.0489...,Compare against E003/E004; see Section 9-10 fo...
2,E003,2026-08-17,LGBMClassifier,raw_predictors,LightGBM should capture the same nonlinear str...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"LGBMClassifier(objective=binary, metric=auc, n...",0.96036,0.95927,0.96133,0.96235,0.96201,0.96106,0.00113,NaN,NaN,CV mean 0.96106 vs E001 0.91149 (delta +0.0495...,Compare against E002/E004; see Section 9-10 fo...
3,E004,2026-08-17,XGBClassifier,raw_predictors,XGBoost's regularization and tree-building beh...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96308,0.96383,0.96403,0.96474,0.96344,0.96382,0.00056,NaN,NaN,CV mean 0.96382 vs E001 0.91149 (delta +0.0523...,Compare against E002/E003; see Section 9-10 fo...


## 8. Benchmark comparison table

In [8]:
results = {
    "E001": {"model": "LogisticRegression", "cv_mean": E001_CV_MEAN, "cv_std": E001_CV_STD,
             "fold_min": None, "fold_max": None, "public_lb": E001_PUBLIC_LB,
             "training_notes": "reference, read from experiments.csv, not rerun"},
    "E002": {"model": "CatBoostClassifier", "cv_mean": e002_result.cv_mean, "cv_std": e002_result.cv_std,
             "fold_min": min(e002_result.fold_scores), "fold_max": max(e002_result.fold_scores),
             "public_lb": None,
             "training_notes": f"{e002_elapsed:.0f}s total, best_iterations={e002_result.best_iterations}"},
    "E003": {"model": "LGBMClassifier", "cv_mean": e003_result.cv_mean, "cv_std": e003_result.cv_std,
             "fold_min": min(e003_result.fold_scores), "fold_max": max(e003_result.fold_scores),
             "public_lb": None,
             "training_notes": f"{e003_elapsed:.0f}s total, best_iterations={e003_result.best_iterations}"},
    "E004": {"model": "XGBClassifier", "cv_mean": e004_result.cv_mean, "cv_std": e004_result.cv_std,
             "fold_min": min(e004_result.fold_scores), "fold_max": max(e004_result.fold_scores),
             "public_lb": None,
             "training_notes": f"{e004_elapsed:.0f}s total, best_iterations={e004_result.best_iterations}"},
}

comparison = pd.DataFrame.from_dict(results, orient="index").reset_index(names="experiment_id")
comparison["feature_set"] = "raw_predictors"
comparison = comparison[
    ["experiment_id", "model", "feature_set", "cv_mean", "cv_std", "fold_min", "fold_max",
     "public_lb", "training_notes"]
]
comparison.to_csv(OUTPUTS_DIR / "model_benchmarks.csv", index=False)
comparison

,experiment_id,model,feature_set,cv_mean,cv_std,fold_min,fold_max,public_lb,training_notes
0,E001,LogisticRegression,raw_predictors,0.911490,0.000810,NaN,NaN,0.91358,"reference, read from experiments.csv, not rerun"
1,E002,CatBoostClassifier,raw_predictors,0.960398,0.000512,0.959820,0.961302,NaN,"2663s total, best_iterations=[799, 799, 799, 7..."
2,E003,LGBMClassifier,raw_predictors,0.961065,0.001126,0.959269,0.962350,NaN,"142s total, best_iterations=[388, 190, 322, 43..."
3,E004,XGBClassifier,raw_predictors,0.963825,0.000560,0.963084,0.964738,NaN,"572s total, best_iterations=[799, 794, 799, 79..."


In [9]:
comparison_display = comparison.copy()
comparison_display["delta_vs_e001"] = comparison_display["cv_mean"] - E001_CV_MEAN
comparison_display[["experiment_id", "model", "cv_mean", "cv_std", "delta_vs_e001", "public_lb"]]

,experiment_id,model,cv_mean,cv_std,delta_vs_e001,public_lb
0,E001,LogisticRegression,0.911490,0.000810,0.000000,0.91358
1,E002,CatBoostClassifier,0.960398,0.000512,0.048908,NaN
2,E003,LGBMClassifier,0.961065,0.001126,0.049575,NaN
3,E004,XGBClassifier,0.963825,0.000560,0.052335,NaN


## 9. OOF prediction correlation diagnostic

Diagnostic only — not used to build an ensemble in this build. Measures
whether the model families are producing similar or materially different
rankings.

In [10]:
oof_frame = pd.DataFrame({
    "E001": np.nan,  # E001's OOF predictions were not persisted in Build 2; excluded here
    "E002": e002_result.oof_predictions,
    "E003": e003_result.oof_predictions,
    "E004": e004_result.oof_predictions,
})
oof_frame = oof_frame.drop(columns=["E001"])  # placeholder column, see note below

oof_correlation = oof_frame.corr()
oof_correlation.to_csv(OUTPUTS_DIR / "oof_prediction_correlation.csv")
oof_correlation

,E002,E003,E004
E002,1.000000,0.991943,0.987900
E003,0.991943,1.000000,0.990053
E004,0.987900,0.990053,1.000000


**Note:** E001's out-of-fold predictions were not persisted when Build 2
ran (`notebooks/02_baseline.ipynb` only recorded fold scores, not OOF
arrays), so the E001 column is excluded from this correlation table rather
than approximated. E002-E004 correlations are exact and comparable to each
other.

## 10. Submission recommendation

Per Build 3 policy: do not submit all three automatically. Classify each
based on local CV.

In [11]:
print("E001 CV:", round(E001_CV_MEAN, 5), "| public LB:", E001_PUBLIC_LB)
print("E002 (CatBoost) CV:", round(e002_result.cv_mean, 5),
      "| delta vs E001:", round(e002_result.cv_mean - E001_CV_MEAN, 5))
print("E003 (LightGBM) CV:", round(e003_result.cv_mean, 5),
      "| delta vs E001:", round(e003_result.cv_mean - E001_CV_MEAN, 5))
print("E004 (XGBoost) CV:", round(e004_result.cv_mean, 5),
      "| delta vs E001:", round(e004_result.cv_mean - E001_CV_MEAN, 5))

E001 CV: 0.91149 | public LB: 0.91358
E002 (CatBoost) CV: 0.9604 | delta vs E001: 0.04891
E003 (LightGBM) CV: 0.96106 | delta vs E001: 0.04957
E004 (XGBoost) CV: 0.96382 | delta vs E001: 0.05233


Classification rule (applied programmatically below, not by eye): a
benchmark is **definitely worth submitting** if it is the single best CV
mean among E002-E004, or if it clearly beats E001 (CV mean more than
3 x E001's fold std above E001's CV mean — using E001's own fold-to-fold
noise as the materiality bar) **and** its out-of-fold predictions are the
least correlated with the best model's (i.e. it adds the most diversity
among the models that clearly beat E001). Everything else is **not
prioritized for submission** in this build — CV already tells us it is
dominated or redundant.

In [12]:
materiality_bar = E001_CV_MEAN + 3 * E001_CV_STD
print(f"materiality bar (E001 CV mean + 3x E001 CV std): {materiality_bar:.5f}")

candidates = {
    "E002": e002_result,
    "E003": e003_result,
    "E004": e004_result,
}
cv_means = {k: v.cv_mean for k, v in candidates.items()}
best_id = max(cv_means, key=cv_means.get)
print(f"best benchmark by CV mean: {best_id} ({cv_means[best_id]:.5f})")

clearly_beats_e001 = {
    k: v for k, v in candidates.items() if v.cv_mean > materiality_bar
}
print("clearly beat E001 by the materiality bar:", list(clearly_beats_e001.keys()))

second_id = None
remaining = {k: v for k, v in clearly_beats_e001.items() if k != best_id}
if remaining:
    best_oof = candidates[best_id].oof_predictions
    oof_corr_to_best = {
        k: np.corrcoef(best_oof, v.oof_predictions)[0, 1] for k, v in remaining.items()
    }
    second_id = min(oof_corr_to_best, key=oof_corr_to_best.get)
    print(f"second most diverse benchmark clearly beating E001: {second_id} "
          f"(OOF corr to {best_id}: {oof_corr_to_best[second_id]:.4f})")
else:
    print("no second candidate clearly beats E001 independently of the best model")

submit_experiment_ids = [best_id] + ([second_id] if second_id else [])
print("recommended for submission:", submit_experiment_ids)

materiality bar (E001 CV mean + 3x E001 CV std): 0.91392
best benchmark by CV mean: E004 (0.96382)
clearly beat E001 by the materiality bar: ['E002', 'E003', 'E004']
second most diverse benchmark clearly beating E001: E002 (OOF corr to E004: 0.9879)
recommended for submission: ['E004', 'E002']


### Generating and validating submission files

Per the Build 3 test-prediction strategy: average the five fold-trained
models' test-set probabilities (already computed by
`run_cv_benchmark`'s `test_predictions`), do not threshold, and do not
retrain a separate full-data model. Each file is validated against
`sample_submission.csv` before being written, and `experiments.csv`'s
`submission_file` column is updated for whichever experiments are
submitted.

In [13]:
def update_submission_file_field(experiment_id: str, filename: str) -> None:
    df = pd.read_csv(experiments_path)
    df["submission_file"] = df["submission_file"].astype(object)
    df.loc[df["experiment_id"] == experiment_id, "submission_file"] = filename
    df.to_csv(experiments_path, index=False)


DELIVERABLES_DIR.mkdir(exist_ok=True)
generated_submissions = {}

for exp_id in submit_experiment_ids:
    result = candidates[exp_id]
    submission = pd.DataFrame({
        ID_COLUMN: test[ID_COLUMN],
        TARGET_COLUMN: result.test_predictions,
    })
    validate_submission(submission, sample_submission)

    filename = f"deliverables/{exp_id}_submission.csv"
    output_path = DELIVERABLES_DIR / f"{exp_id}_submission.csv"
    submission.to_csv(output_path, index=False)
    update_submission_file_field(exp_id, filename)
    generated_submissions[exp_id] = filename
    print(f"{exp_id}: wrote {len(submission)} rows to {output_path}, validated OK")

print()
print("Generated submissions:", generated_submissions)
print("These are ready for manual Kaggle submission. Public LB scores will be")
print("recorded against their experiment rows once provided.")

E004: wrote 296302 rows to D:\Projects\kaggle-smartphone-addiction\deliverables\E004_submission.csv, validated OK


E002: wrote 296302 rows to D:\Projects\kaggle-smartphone-addiction\deliverables\E002_submission.csv, validated OK

Generated submissions: {'E004': 'deliverables/E004_submission.csv', 'E002': 'deliverables/E002_submission.csv'}
These are ready for manual Kaggle submission. Public LB scores will be
recorded against their experiment rows once provided.


## 11. Computational notes

- CatBoost: capped at the shared 800-iteration budget; best iteration was
  still the final iteration in every fold (early stopping did not
  trigger), meaning CatBoost had not fully converged within this budget.
  This is a deliberate runtime-practicality tradeoff (an initial 2000-
  iteration timing check showed CatBoost still improving with no sign of
  plateauing), not a claim that 800 iterations is CatBoost's ceiling.
- LightGBM: fastest of the three; early stopping triggered well before
  the 800-iteration cap in initial timing checks (~300 iterations),
  suggesting it converges efficiently on this dataset.
- XGBoost: also ran close to the 800-iteration cap without triggering
  early stopping in initial timing checks, similar to CatBoost.

Exact per-model elapsed time and best_iterations for the real 5-fold runs
are in Section 8's comparison table and `outputs/model_benchmarks.csv`.

## 12. Build 3 conclusions

Answers computed from the actual results above, not asserted in advance.

In [14]:
print("1. Did all boosters beat Logistic Regression?")
for exp_id, result in candidates.items():
    beat = result.cv_mean > E001_CV_MEAN
    print(f"   {exp_id}: {'YES' if beat else 'NO'} "
          f"(cv_mean={result.cv_mean:.5f} vs E001={E001_CV_MEAN:.5f}, "
          f"delta={result.cv_mean - E001_CV_MEAN:+.5f})")

print()
print("2. Which booster achieved the best mean CV?")
print(f"   {best_id}: cv_mean={cv_means[best_id]:.5f}, "
      f"delta vs E001 = {cv_means[best_id] - E001_CV_MEAN:+.5f}")

print()
print("3. Is the improvement large relative to fold variance?")
best_result = candidates[best_id]
delta = best_result.cv_mean - E001_CV_MEAN
combined_std = (best_result.cv_std ** 2 + E001_CV_STD ** 2) ** 0.5
print(f"   delta={delta:.5f}, combined fold std (quadrature)={combined_std:.5f}, "
      f"ratio={delta / combined_std:.1f}x" if combined_std > 0 else "   combined std is 0")

print()
print("4. Which model had the lowest fold variance?")
stds = {k: v.cv_std for k, v in candidates.items()}
lowest_std_id = min(stds, key=stds.get)
print(f"   {lowest_std_id}: cv_std={stds[lowest_std_id]:.5f} (lowest fold-to-fold variance among E002-E004)")

print()
print("5. Did early-stopping behavior differ materially across folds?")
for exp_id, result in candidates.items():
    if result.best_iterations:
        spread = max(result.best_iterations) - min(result.best_iterations)
        print(f"   {exp_id}: best_iterations={result.best_iterations}, spread={spread}")

print()
print("6. How do CV and Public LB compare?")
print(f"   E001: CV={E001_CV_MEAN:.5f}, public LB={E001_PUBLIC_LB} "
      f"(delta={E001_PUBLIC_LB - E001_CV_MEAN:+.5f})" if E001_PUBLIC_LB is not None
      else "   E001: no public LB recorded")
print("   E002-E004: not yet submitted in this build (see Section 10) or",
      "awaiting a public LB result to compare — recorded once provided.")

print()
print("7. Which model becomes the primary Build 4 control?")
print(f"   {best_id} — highest CV mean ({cv_means[best_id]:.5f}) under the shared harness.")

print()
print("8. Should another model remain a secondary control?")
if second_id:
    print(f"   {second_id} — clearly beats E001 and is the most diverse (lowest OOF "
          f"correlation to {best_id}) among the remaining candidates.")
else:
    print("   No second candidate met the materiality + diversity bar; revisit if "
          "Build 4 feature engineering narrows the gap between models.")

1. Did all boosters beat Logistic Regression?
   E002: YES (cv_mean=0.96040 vs E001=0.91149, delta=+0.04891)
   E003: YES (cv_mean=0.96106 vs E001=0.91149, delta=+0.04957)
   E004: YES (cv_mean=0.96382 vs E001=0.91149, delta=+0.05233)

2. Which booster achieved the best mean CV?
   E004: cv_mean=0.96382, delta vs E001 = +0.05233

3. Is the improvement large relative to fold variance?
   delta=0.05233, combined fold std (quadrature)=0.00098, ratio=53.1x

4. Which model had the lowest fold variance?
   E002: cv_std=0.00051 (lowest fold-to-fold variance among E002-E004)

5. Did early-stopping behavior differ materially across folds?
   E002: best_iterations=[799, 799, 799, 799, 799], spread=0
   E003: best_iterations=[388, 190, 322, 439, 634], spread=444
   E004: best_iterations=[799, 794, 799, 794, 794], spread=5

6. How do CV and Public LB compare?
   E001: CV=0.91149, public LB=0.91358 (delta=+0.00209)
   E002-E004: not yet submitted in this build (see Section 10) or awaiting a public 